In [25]:
import pandas as pd
import numpy as np
import random


road_types = ["Highway", "Main Road", "Residential"]
weather_conditions = ["Clear", "Rain", "Fog", "Storm", "Cloudy"]
accident_occurrence = ["Yes", "No"]
time_of_day = ["Morning", "Afternoon", "Evening", "Night"]
congestion_levels = ["Low", "Medium", "High"]

def get_congestion(vehicle_count, avg_speed, wait_time):
    if vehicle_count > 120 or wait_time > 90 or avg_speed < 25:
        return "High"
    elif 50 <= vehicle_count <= 120 or 30 <= wait_time <= 90:
        return "Medium"
    else:
        return "Low"


data = []
for _ in range(100):
    vc = random.randint(10, 200)
    spd = round(random.uniform(15, 65), 1)
    wait = random.randint(10, 150)
    road = random.choice(road_types)
    weather = random.choice(weather_conditions)
    accident = random.choice(accident_occurrence)
    tod = random.choice(time_of_day)
    congestion = get_congestion(vc, spd, wait)

    data.append([vc, spd, wait, road, weather, accident, tod, congestion])


df = pd.DataFrame(data, columns=[
    "vehicle_count", "average_speed_kmph", "traffic_signal_wait_sec",
    "road_type", "weather_conditions", "accident_occurrence",
    "time_of_day", "traffic_congestion"
])

df.to_csv("urban_traffic_dataset.csv", index=False)
print("Dataset saved as urban_traffic_dataset.csv")



Dataset saved as urban_traffic_dataset.csv


In [26]:
df.head(10)

,vehicle_count,average_speed_kmph,traffic_signal_wait_sec,road_type,weather_conditions,accident_occurrence,time_of_day,traffic_congestion
0,96,52.6,92,Main Road,Rain,Yes,Morning,High
1,60,38.4,24,Highway,Storm,Yes,Morning,Medium
2,79,63.8,56,Residential,Cloudy,Yes,Evening,Medium
3,128,53.6,37,Main Road,Rain,Yes,Morning,High
4,129,23.1,147,Highway,Fog,No,Afternoon,High
5,184,65.0,109,Residential,Fog,No,Afternoon,High
6,54,23.5,110,Main Road,Fog,Yes,Morning,High
7,40,60.3,98,Residential,Clear,Yes,Night,High
8,16,51.2,17,Residential,Clear,Yes,Afternoon,Low
9,117,17.6,51,Highway,Rain,Yes,Morning,High


In [27]:
y=df['traffic_congestion']

In [28]:
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

scaler = MinMaxScaler()
df['vehicle_count'] = scaler.fit_transform(df[['vehicle_count']])
print("Normalized 'vehicle_count':")
print(df.head())


target_encoded_cols = [col for col in df.columns if col.startswith('traffic_congestion_')]
if target_encoded_cols:
    df = df.drop(columns=target_encoded_cols)


categorical_feature_cols = df.select_dtypes(include='object').columns
if not categorical_feature_cols.empty:
    df = pd.get_dummies(df, columns=categorical_feature_cols, drop_first=True)

print("\nDataFrame after ensuring it contains only feature columns (df is now X):")
df.drop(columns=['traffic_congestion_Low', 'traffic_congestion_Medium'], inplace=True)
print(df.head())

Normalized 'vehicle_count':
   vehicle_count  average_speed_kmph  traffic_signal_wait_sec    road_type  \
0       0.449198                52.6                       92    Main Road   
1       0.256684                38.4                       24      Highway   
2       0.358289                63.8                       56  Residential   
3       0.620321                53.6                       37    Main Road   
4       0.625668                23.1                      147      Highway   

  weather_conditions accident_occurrence time_of_day traffic_congestion  
0               Rain                 Yes     Morning               High  
1              Storm                 Yes     Morning             Medium  
2             Cloudy                 Yes     Evening             Medium  
3               Rain                 Yes     Morning               High  
4                Fog                  No   Afternoon               High  

DataFrame after ensuring it contains only feature columns 

In [29]:
df.columns


Index(['vehicle_count', 'average_speed_kmph', 'traffic_signal_wait_sec',
       'road_type_Main Road', 'road_type_Residential',
       'weather_conditions_Cloudy', 'weather_conditions_Fog',
       'weather_conditions_Rain', 'weather_conditions_Storm',
       'accident_occurrence_Yes', 'time_of_day_Evening', 'time_of_day_Morning',
       'time_of_day_Night'],
      dtype='object')

In [30]:

df.isnull().sum()

,0
vehicle_count,0
average_speed_kmph,0
traffic_signal_wait_sec,0
road_type_Main Road,0
road_type_Residential,0
weather_conditions_Cloudy,0
weather_conditions_Fog,0
weather_conditions_Rain,0
weather_conditions_Storm,0
accident_occurrence_Yes,0


In [31]:
from sklearn.model_selection import train_test_split
import pandas as pd


feature_cols = [col for col in df.columns if not col.startswith('traffic_congestion_')]
X = df[feature_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train)
print(y_train)

    vehicle_count  average_speed_kmph  traffic_signal_wait_sec  \
55       0.069519                44.8                       26   
88       0.529412                46.0                       82   
26       0.518717                48.5                      136   
42       0.791444                46.7                       67   
69       0.780749                64.5                      106   
..            ...                 ...                      ...   
60       0.000000                60.3                       85   
71       0.272727                37.0                       15   
14       0.780749                25.3                       68   
92       0.556150                56.5                       77   
51       0.133690                42.9                       27   

    road_type_Main Road  road_type_Residential  weather_conditions_Cloudy  \
55                 True                  False                      False   
88                 True                  False       

In [44]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


rf_model = RandomForestClassifier(random_state=42, n_estimators=250, max_depth=5)

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)


accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

Accuracy: 0.8500
Precision: 0.8082
Recall: 0.8500
F1-Score: 0.8242


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [45]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


knn= KNeighborsClassifier()

knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

Accuracy: 0.6000
Precision: 0.6536
Recall: 0.6000
F1-Score: 0.5960
